# Skeleton-of-thought (SoT)

Language models generate text token by token, in a single sequential pass. When a question requires covering several distinct aspects - causes, effects, categories, trade-offs - the model has to hold all of them in mind simultaneously while writing, which often leads to uneven coverage: some aspects receive paragraphs while others get a single sentence, and some are skipped entirely.

Skeleton-of-Thought addresses this by splitting the task in two. In the first phase, the model produces only a *skeleton* - an ordered list of headings that define what the final answer should cover. In the second phase, each heading is expanded independently into its own paragraph. Because the expansions are independent of each other, they can be dispatched in parallel threads, which brings the total latency down to roughly the time of the slowest single expansion rather than the sum of all of them.

The two benefits - structural completeness and parallel speed - are separable. A sequential SoT still guarantees coverage via the skeleton checklist, it just forgoes the speed benefit. A direct parallel call without a skeleton would be faster but would lose the structural guarantee. This notebook implements both phases step by step and then measures both effects.

In [1]:
import os
import time  # Measuring wall-clock latency for both expansion modes
from concurrent.futures import ThreadPoolExecutor  # Dispatching independent expansion tasks in parallel
from typing import List

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

### Initialize the language model

In [2]:
# temperature=0 keeps the skeleton deterministic; expansions benefit from
# a little variation so they don't all sound identical — 0.7 is a reasonable default
llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY", "").strip(), temperature=0.7)

## Phase 1 - Generating the skeleton
The skeleton generation call is intentionally minimal: we ask for headings only, no content. This is important because we want the model to plan the full structure of the answer *before* it is committed to any content. If we asked for headings and content together, the model would start filling in the first section and potentially lose track of later ones - which is exactly the problem SoT is designed to avoid. Requesting a numbered list also gives us a reliable parse target: any line whose first character is a digit followed by a period is a heading.

In [3]:
def generate_skeleton(question: str) -> List[str]:
    """Return an ordered list of headings that form the answer skeleton.

    Each heading names one section to be expanded later. No content is generated here.
    """
    prompt = (
        f"Given the following question, generate a numbered list of main points "
        f"that a comprehensive answer should cover.\n\n"
        f"Question: {question}\n\n"
        f"Rules:\n"
        f"- Each item must be a brief heading only — no explanatory text\n"
        f"- Aim for 4-6 points that together give complete coverage\n"
        f"- Output only the numbered list, nothing else\n\n"
        f"Format:\n"
        f"1. First heading\n"
        f"2. Second heading\n"
        f"..."
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    raw = response.content.strip()

    headings = []
    for line in raw.splitlines():
        line = line.strip()
        # Match lines that start with a digit followed by a period (e.g. "1.", "12.")
        if line and line[0].isdigit() and '.' in line:
            heading = line.split('.', 1)[1].strip()   # text after the "N." prefix
            if heading:
                headings.append(heading)

    return headings

The function returns a plain `List[str]` - just the heading texts in order. There is no index field on a data object because the index is always implicit in the list position (`enumerate` provides it when needed). `split('.', 1)` splits on the first period only, so headings that contain periods (e.g. "U.S. energy policy") are preserved correctly.

In [4]:
# Quick test: verify the skeleton is a clean list of headings
test_question = "What are the key factors contributing to climate change, and what can be done to address it?"
test_headings = generate_skeleton(test_question)

print(f"Question: {test_question}\n")
print(f"Skeleton ({len(test_headings)} points):")
for i, heading in enumerate(test_headings, 1):
    print(f"  {i}. {heading}")

Question: What are the key factors contributing to climate change, and what can be done to address it?

Skeleton (6 points):
  1. Greenhouse Gas Emissions
  2. Deforestation
  3. Fossil Fuel Consumption
  4. Industrial Processes
  5. Agriculture and Land Use
  6. Renewable Energy Solutions


## Phase 2 - Expanding a single point
Each heading gets expanded into a self-contained paragraph that is independent of all the others. The expansion prompt includes the original question for context - without it, the model cannot know whether "Economic Impacts" means impacts on a household, a country, or the global economy. The critical instruction is "write this section only, do not repeat the heading": without this, models often begin each expansion by restating the heading as a sentence, which wastes tokens and creates redundant output in the final assembled answer.

In [5]:
def expand_point(question: str, heading: str) -> str:
    """Expand a single skeleton heading into a detailed paragraph.

    The original question is included so the model knows the full context
    even though it is only writing one isolated section.
    """
    prompt = (
        f"You are writing one section of a comprehensive answer.\n\n"
        f"Original question: {question}\n"
        f"Section heading:   {heading}\n\n"
        f"Write a detailed, focused paragraph for this section only. "
        f"Do not repeat or rephrase the heading in your response — go straight into the content."
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content.strip()

Each call to `expand_point` is fully self-contained: it receives only `question` and `heading`, invokes the LLM independently, and returns a plain string. This isolation is what makes parallel dispatch safe - the calls share no state, so there are no race conditions when multiple threads call this function simultaneously with different headings.

In [6]:
# Quick test: expand the first heading from the skeleton we just generated
if test_headings:
    sample_expansion = expand_point(test_question, test_headings[0])
    print(f"Heading: {test_headings[0]}\n")
    print(f"Expanded content:\n{sample_expansion}")

Heading: Greenhouse Gas Emissions

Expanded content:
Greenhouse gas emissions are the primary driver of climate change, resulting chiefly from human activities such as burning fossil fuels, deforestation, and industrial processes. Carbon dioxide (CO2), methane (CH4), and nitrous oxide (N2O) are the most significant gases contributing to the greenhouse effect, which traps heat in the Earth's atmosphere. The combustion of coal, oil, and natural gas for energy and transportation releases vast amounts of CO2, while agricultural practices and livestock production are major sources of methane and nitrous oxide emissions. To effectively address these emissions, a multi-faceted approach is essential. This includes transitioning to renewable energy sources like solar, wind, and hydroelectric power, enhancing energy efficiency in buildings and transportation, adopting sustainable agricultural practices, and implementing carbon pricing mechanisms to incentivize emission reductions. Additionally, 

## Expanding all points - parallel and sequential
With a function that expands one point in isolation, expanding all points is straightforward. The sequential version is just a list comprehension. The parallel version uses `ThreadPoolExecutor`: all tasks are submitted at once with `executor.submit`, then results are collected with `[f.result() for f in futures]`. That second list comprehension iterates the futures in *submission order* - not completion order - which guarantees that the resulting expansions list aligns with the headings list even when some tasks finish earlier than others.

In [7]:
def expand_all(question: str, headings: List[str],
               parallel: bool = True, max_workers: int = 4) -> List[str]:
    """Expand every heading, returning a list of paragraphs in the same order.

    parallel=True dispatches all expansions at once using a thread pool.
    parallel=False processes them one after another.
    """
    if parallel:
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            # Submit all N expansion tasks simultaneously — they run concurrently
            futures = [executor.submit(expand_point, question, h) for h in headings]
            # Collect results in submission order, not completion order.
            # This ensures expansions[i] always corresponds to headings[i].
            return [f.result() for f in futures]
    else:
        # Sequential: each heading waits for the previous one to finish
        return [expand_point(question, heading) for heading in headings]

The `with ThreadPoolExecutor(...) as executor` block ensures the pool is shut down cleanly after all futures complete - no explicit `.shutdown()` call is needed. `max_workers` caps the number of simultaneous HTTP requests; setting it too high can trigger API rate limits, while setting it lower than the number of headings means some tasks queue rather than run immediately. For typical skeleton sizes of 4-6 points, `max_workers=4` is a safe default.

## Assembling the final answer
Once all headings have been expanded, the assembly step interleaves them into a single readable document. The heading becomes a bold section title, and its expansion follows immediately below. A horizontal rule (`---`) between sections gives visual separation and reinforces the skeleton structure in the output - the reader can see exactly which heading produced which paragraph.

In [8]:
def assemble_answer(headings: List[str], expansions: List[str]) -> str:
    """Combine headings and their expansions into a single formatted answer.

    zip(headings, expansions) pairs each heading with its corresponding paragraph.
    The two lists are always the same length because expand_all preserves submission order.
    """
    sections = [
        f"**{heading}**\n\n{content}"
        for heading, content in zip(headings, expansions)   # pair by position
    ]
    # Join sections with a horizontal rule so the skeleton structure stays visible
    return "\n\n---\n\n".join(sections)

`zip(headings, expansions)` is the correct tool here: it stops at the shorter of the two lists, so a partial failure (e.g. one expansion call returning an empty string) will not cause an index error in the assembly step. The formatted string places the heading in bold (`**...**`) followed by two newlines before the content - standard Markdown paragraph separation that renders cleanly in a Jupyter notebook.

## The complete pipeline
The four functions — `generate_skeleton`, `expand_all`, and `assemble_answer` - are now assembled into a single driver. The driver prints progress at each phase so the two-phase structure stays visible during execution. Timing covers the expansion phase only (the skeleton call is typically fast and consistent); timing both together would conflate the structural planning cost with the parallelism benefit, making the later comparison less informative.

In [9]:
def solve_with_sot(question: str, parallel: bool = True, max_workers: int = 4) -> dict:
    """Full SoT pipeline: generate skeleton → expand all → assemble.

    Returns a dict with all intermediate data so callers can inspect any step.
    """
    print(f"Question: {question}\n")

    # Phase 1 — Plan the answer structure before committing to any content
    print("Phase 1 — Generating skeleton...")
    headings = generate_skeleton(question)
    for i, h in enumerate(headings, 1):
        print(f"  {i}. {h}")

    # Phase 2 — Expand each heading independently; time only this phase
    mode = "parallel" if parallel else "sequential"
    print(f"\nPhase 2 — Expanding {len(headings)} points ({mode})...")
    t0 = time.time()
    expansions = expand_all(question, headings, parallel=parallel, max_workers=max_workers)
    expansion_time = time.time() - t0
    print(f"  Done in {expansion_time:.2f}s")

    # Phase 3 — Interleave headings and expansions into the final document
    final_answer = assemble_answer(headings, expansions)

    return {
        "question":       question,
        "headings":       headings,       # the skeleton — list of heading strings
        "expansions":     expansions,     # one paragraph per heading, same order
        "final_answer":   final_answer,
        "expansion_time": expansion_time,
        "parallel":       parallel,
    }

The driver returns a plain dict so every intermediate value - the headings list, the individual expansion strings, and the assembled answer - is accessible without a custom class. `expansion_time` isolates the cost of the expansion phase, which is the only part that differs between parallel and sequential modes. The skeleton and assembly costs are identical in both cases.

## Applying the pipeline to a comprehensive question
Let's run the full pipeline on a multi-faceted question and inspect both the skeleton and the assembled answer. A climate-change question is a good test because it genuinely has several independent sub-topics (causes, effects, policy responses, individual action) that benefit from the structure a skeleton provides.

In [10]:
question = (
    "What are the key factors contributing to climate change, "
    "and what can be done to address it?"
)

result = solve_with_sot(question, parallel=True)

print("\n" + "=" * 65)
print("SKELETON")
print("=" * 65)
for i, h in enumerate(result["headings"], 1):
    print(f"  {i}. {h}")

print("\n" + "=" * 65)
print("ASSEMBLED ANSWER")
print("=" * 65)
print(result["final_answer"])

Question: What are the key factors contributing to climate change, and what can be done to address it?

Phase 1 — Generating skeleton...
  1. Greenhouse Gas Emissions
  2. Deforestation
  3. Industrial Activities
  4. Agriculture and Land Use
  5. Renewable Energy Solutions
  6. Policy and International Cooperation

Phase 2 — Expanding 6 points (parallel)...
  Done in 8.28s

SKELETON
  1. Greenhouse Gas Emissions
  2. Deforestation
  3. Industrial Activities
  4. Agriculture and Land Use
  5. Renewable Energy Solutions
  6. Policy and International Cooperation

ASSEMBLED ANSWER
**Greenhouse Gas Emissions**

Greenhouse gas emissions are primarily driven by human activities, particularly the burning of fossil fuels such as coal, oil, and natural gas for energy production, transportation, and industrial processes. These emissions release carbon dioxide (CO2), methane (CH4), and nitrous oxide (N2O) into the atmosphere, significantly enhancing the greenhouse effect and leading to global war

The skeleton acts as a checklist: every item in `result["headings"]` is guaranteed to have a corresponding paragraph in the final answer. In a direct one-shot generation, the model might cover some topics briefly and skip others entirely. Here, the skeleton locks in which topics must appear before any content is written.

## Parallel vs sequential - measuring the speed benefit
The speed comparison must be conducted fairly: the skeleton should be generated *once* and shared between both runs, so we are measuring only the expansion phase. Generating the skeleton twice (once per run) would add noise from an additional LLM call that has nothing to do with the parallelism being measured. Using the same headings also ensures both runs produce expansions for the same content, making the timing directly comparable.

In [11]:
# Generate the skeleton once — used for both expansion modes
timing_question = "Explain the main stages of the software development lifecycle."
print(f"Question: {timing_question}\n")
print("Generating skeleton (shared for both runs)...")
shared_headings = generate_skeleton(timing_question)
print(f"Skeleton: {len(shared_headings)} points\n")
for i, h in enumerate(shared_headings, 1):
    print(f"  {i}. {h}")

# Measure parallel expansion on the shared skeleton
print("\n--- Parallel expansion ---")
t0 = time.time()
expansions_par = expand_all(timing_question, shared_headings, parallel=True, max_workers=4)
t_par = time.time() - t0
print(f"  Completed in {t_par:.2f}s")

# Measure sequential expansion on the same skeleton
print("\n--- Sequential expansion ---")
t0 = time.time()
expansions_seq = expand_all(timing_question, shared_headings, parallel=False)
t_seq = time.time() - t0
print(f"  Completed in {t_seq:.2f}s")

# Report the speedup
print(f"\nSpeedup: {t_seq / t_par:.1f}x  "
      f"({len(shared_headings)} points, up to 4 concurrent workers)")

Question: Explain the main stages of the software development lifecycle.

Generating skeleton (shared for both runs)...
Skeleton: 6 points

  1. Requirement Analysis
  2. Design
  3. Implementation
  4. Testing
  5. Deployment
  6. Maintenance

--- Parallel expansion ---
  Completed in 6.36s

--- Sequential expansion ---
  Completed in 22.21s

Speedup: 3.5x  (6 points, up to 4 concurrent workers)


Both runs expand the exact same headings, so the only variable is whether requests are dispatched concurrently or one at a time. The theoretical maximum speedup is `n_points` (e.g. 5× for five points with unlimited workers), but practical speedup is lower because API response times vary and some tasks still queue when the thread pool is saturated. Values of 2-4× are typical for 4-6 point skeletons with `max_workers=4`.

## Comparing with direct generation
To quantify the structural benefit - separate from the speed benefit - we compare SoT against a single direct LLM call on the same question. The direct call is faster (one round trip vs. many) but gives the model no checklist to follow; coverage depends entirely on what the model spontaneously includes. SoT trades latency for the guarantee that every heading in the skeleton will appear in the final answer.

In [12]:
def direct_answer(question: str) -> str:
    """Single LLM call — no skeleton, no parallelism."""
    response = llm.invoke([HumanMessage(
        content=f"Provide a comprehensive answer to: {question}"
    )])
    return response.content.strip()

comparison_q = "What are the main benefits and challenges of renewable energy?"

print("--- Direct answer (1 LLM call) ---")
t0 = time.time()
direct = direct_answer(comparison_q)
t_direct = time.time() - t0
# Show only the first 300 characters so output stays readable
print(direct[:300] + "...")
print(f"\nTime: {t_direct:.2f}s")

print("\n" + "-" * 65)
print("--- Skeleton-of-Thought ---")
sot_result = solve_with_sot(comparison_q, parallel=True)

print("\nSkeleton:")
for i, h in enumerate(sot_result["headings"], 1):
    print(f"  {i}. {h}")

print("\nFirst section of assembled answer:")
# Display the first section only to keep output manageable
first_section = sot_result["final_answer"].split("---")[0].strip()
print(first_section[:300] + "...")
print(f"\nExpansion time: {sot_result['expansion_time']:.2f}s "
      f"({len(sot_result['headings'])} LLM calls in parallel)")

--- Direct answer (1 LLM call) ---
Renewable energy refers to energy derived from resources that are naturally replenished, such as sunlight, wind, rain, tides, waves, and geothermal heat. The transition from fossil fuels to renewable energy sources has gained significant momentum in recent years due to increasing concerns about clim...

Time: 22.54s

-----------------------------------------------------------------
--- Skeleton-of-Thought ---
Question: What are the main benefits and challenges of renewable energy?

Phase 1 — Generating skeleton...
  1. Environmental Benefits
  2. Economic Advantages
  3. Energy Security
  4. Technological Innovation
  5. Infrastructure Challenges
  6. Intermittency Issues

Phase 2 — Expanding 6 points (parallel)...
  Done in 7.14s

Skeleton:
  1. Environmental Benefits
  2. Economic Advantages
  3. Energy Security
  4. Technological Innovation
  5. Infrastructure Challenges
  6. Intermittency Issues

First section of assembled answer:
**Environmental 

**When to use Skeleton-of-Thought:** Questions that genuinely require covering several distinct, independent sub-topics - comprehensive explanations, comparative analyses, multi-step processes. It is less useful for questions with a single focused answer, where the skeleton overhead adds latency without adding coverage. Sequential SoT (no parallelism) still guarantees structural coverage - every heading in the skeleton becomes a section in the final answer. Parallel SoT adds speed on top of that. Choosing between them is a trade-off between latency and API rate-limit pressure, not between correctness and incorrectness.

**Limitations:** SoT adds at least `n_headings + 1` LLM calls per question (one skeleton call plus one per heading); for short, focused questions this overhead is unnecessary. The assembled answer reads as structured sections rather than flowing prose - if narrative continuity matters, a post-processing pass that rewrites the sections as a single cohesive essay may be worth adding.